# Automated condition-response scoring

Apply keyword and fuzzy matching to condition responses and assess agreement with the manual coding.

## Load packages and configure paths

In [ ]:
import re
import unicodedata
from dataclasses import dataclass
from enum import Enum
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz, process

ROOT = Path("..").resolve()
DATA_PATH = ROOT / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"

VIGNETTES = ["A", "B", "C", "D", "E"]
PHASES = ["pre", "post"]
ID_COL = "prolific_id"

TEXT_COLS = {(v, phase): f"vig_{v}_{phase}_condition" for v in VIGNETTES for phase in PHASES}
MANUAL_COLS = {(v, phase): f"vig_{v}_{phase}_condition_correct" for v in VIGNETTES for phase in PHASES}
FUZZY_COLS = {(v, phase): f"vig_{v}_{phase}_condition_fuzzy_correct" for v in VIGNETTES for phase in PHASES}


## Define accepted and rejected terms

In [ ]:
"""Condition diagnosis lists per vignette (Cases A-E).

Encodes the expert-panel coding guide (majority >=4/6 rule; split-decisions
resolved by conservative default for diagnosis). Accepted = terms endorsed by
majority; rejected = terms explicitly voted down or excluded by the panel, OR
terms that must be actively blocked so the fuzzy/keyword matcher does not
mis-accept them.

Join rule: and / with / plus / +  ->  an accepted clause combined with any
non-accepted clause is scored INCORRECT (participant did not commit to a single
correct identification). OR / comma / slash -> any single accepted alternative
is sufficient (lenient).

Longest-keyword-match wins; ties favour rejected terms (conservative).
"""

from __future__ import annotations

CASE_CONFIG: dict[str, dict[str, list[str]]] = {
    # -----------------------------------------------------------------
    # Case A — Psychosis
    # Accepted (guide): psychosis, first-episode psychosis, schizophrenia,
    #   schizoaffective disorder, psychotic disorder, prodromal phase of
    #   psychotic disorder / schizophrenia
    # Rejected (guide): MDD with psychotic features (1/6); paranoia alone
    #   (4/6 against); depression/anxiety/stress without psychotic component;
    #   burnout/adjustment disorder
    # -----------------------------------------------------------------
    "A": {
        "accepted": [
            "psychosis",
            "first-episode psychosis",
            "first episode psychosis",
            "schizophrenia",
            "schizoaffective disorder",
            "psychotic disorder",
            "prodromal psychosis",
            "prodromal phase of psychotic disorder",
            "prodromal schizophrenia",
        ],
        "rejected": [
            "paranoia",                       # 4/6 voted against; alone not accepted
            "paranoid",
            "mdd with psychotic features",    # 1/6 only
            "major depressive disorder with psychotic features",
            "depression",
            "anxiety",
            "stress",
            "burnout",
            "adjustment disorder",
        ],
    },

    # -----------------------------------------------------------------
    # Case B — Panic disorder
    # Accepted (guide): panic disorder, panic attacks, anxiety attacks
    # Rejected (guide): anxiety alone (0/6); anxiety disorder (0/6);
    #   agoraphobia alone (3/6 split, conservative default); health/illness
    #   anxiety (1/6); somatic symptom disorder (1/6); cardiac/heart condition
    # NOTE: "anxiety attacks" is accepted but contains "anxiety" (rejected);
    #   longest-match tie-break ensures "anxiety attacks" wins over "anxiety".
    # -----------------------------------------------------------------
    "B": {
        "accepted": [
            "panic disorder",
            "panic attack",
            "panic attacks",
            "anxiety attack",
            "anxiety attacks",
        ],
        "rejected": [
            "anxiety disorder",
            "anxiety",
            "generalised anxiety",
            "generalized anxiety",
            "gad",
            "agoraphobia",
            "health anxiety",
            "illness anxiety",
            "somatic symptom disorder",
            "somatic symptom",
            "heart condition",
            "cardiac",
            "heart problem",
        ],
    },

    # -----------------------------------------------------------------
    # Case C — Obsessive-compulsive disorder
    # Accepted (guide): OCD, obsessive-compulsive disorder (post-viral /
    #   secondary specifier also accepted)
    # Rejected (guide): perfectionism alone; anxiety/GAD; burnout/stress
    # -----------------------------------------------------------------
    "C": {
        "accepted": [
            "ocd",
            "obsessive-compulsive disorder",
            "obsessive compulsive disorder",
            "obsessive compulsive",
            "obsessive-compulsive",
        ],
        "rejected": [
            "perfectionism",
            "perfectionist",
            "anxiety",
            "generalised anxiety",
            "generalized anxiety",
            "gad",
            "burnout",
            "stress",
        ],
    },

    # -----------------------------------------------------------------
    # Case D — Mania / bipolar disorder
    # Accepted (guide): mania, manic episode, bipolar disorder, bipolar I,
    #   first episode mania, bipolar disorder with manic episode
    # Rejected (guide): substance-induced mania (excluded by vignette - no
    #   drug history); stress/burnout/overwork
    # NOTE: "substance-induced mania" contains "mania"; longest-match
    #   tie-break ensures the rejected phrase wins.
    # -----------------------------------------------------------------
    "D": {
        "accepted": [
            "mania",
            "manic episode",
            "manic",
            "bipolar disorder",
            "bipolar i",
            "bipolar 1",
            "bipolar",
            "first episode mania",
            "bipolar disorder with manic episode",
        ],
        "rejected": [
            "substance-induced mania",
            "substance induced mania",
            "drug-induced mania",
            "drug induced mania",
            "stress",
            "burnout",
            "overwork",
            "overworked",
        ],
    },

    # -----------------------------------------------------------------
    # Case E — Normal stress response / no disorder
    # Accepted (guide): no diagnosis / nothing wrong / normal response,
    #   normal adjustment to a new job, stress, work-related stress
    # Rejected (guide): adjustment DISORDER (3/6 split, conservative default);
    #   burnout (0/6); anxiety/depression (over-pathologising)
    # CRITICAL: this is the over-pathologising guard. "adjustment" is NOT
    #   accepted on its own as a disorder; only "normal adjustment" is the
    #   intended accepted sense. We accept the normal-response terms and
    #   actively reject any named clinical disorder so that e.g.
    #   "stress and burnout" or "adjustment disorder" scores incorrect.
    # -----------------------------------------------------------------
    "E": {
        "accepted": [
            "no diagnosis",
            "no disorder",
            "no condition",
            "no illness",
            "nothing wrong",
            "nothing",
            "normal response",
            "normal adjustment",
            "normal",
            "stress",
            "work-related stress",
            "work related stress",
            "stressed",
            "overworked",
            "fine",
            "healthy",
        ],
        "rejected": [
            "adjustment disorder",
            "burnout",
            "anxiety",
            "anxiety disorder",
            "depression",
            "depressive disorder",
            "stress disorder",
            "acute stress disorder",
            "ptsd",
            "post-traumatic stress",
            "post traumatic stress",
        ],
    },
}

## Define scoring logic

In [ ]:
"""Fuzzy condition-diagnosis scoring with join/OR rules."""

DEFAULT_ACCEPT_THRESHOLD = 82
DEFAULT_REJECT_THRESHOLD = 82
DEFAULT_SCORER = fuzz.token_sort_ratio

LEADING_FILLER_PATTERNS = [
    r"^likely condition\s*:?\s*",
    r"^possible condition\s*:?\s*",
    r"^i think\s+(?:this person is\s+)?(?:suffering from\s+)?",
    r"^i believe\s+",
    r"^probably\s+",
    r"^possibly\s+",
    r"^maybe\s+",
    r"^seems like\s+",
]

COMBINER_RE = re.compile(r"\band\b|\bwith\b|\bplus\b|\+", re.IGNORECASE)


class ClauseLabel(str, Enum):
    ACCEPTED = "accepted"
    REJECTED = "rejected"
    UNKNOWN = "unknown"


@dataclass
class ClauseMatch:
    clause: str
    label: ClauseLabel
    score: float
    matched_term: str | None


@dataclass
class ConditionScoreResult:
    fuzzy_correct: int | None
    best_score: float
    matched_term: str | None
    rule: str
    clause_matches: list[ClauseMatch]


def normalize_text(text: object) -> str:
    if text is None or (isinstance(text, float) and __import__("pandas").isna(text)):
        return ""

    s = unicodedata.normalize("NFKD", str(text)).strip().lower()
    s = s.replace("\u2011", "-").replace("\u2013", "-").replace("\u2014", "-")
    s = re.sub(r"[^\w\s\-/(),.?']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    for pattern in LEADING_FILLER_PATTERNS:
        s = re.sub(pattern, "", s, flags=re.IGNORECASE).strip()

    return s


def _phrase_in_text(phrase: str, text: str) -> bool:
    phrase = normalize_text(phrase)
    if not phrase or not text:
        return False
    if " " in phrase:
        return phrase in text
    return re.search(rf"\b{re.escape(phrase)}\b", text) is not None


def _best_term_match(clause: str, terms: list[str], scorer=DEFAULT_SCORER) -> tuple[str | None, float]:
    normalized_clause = normalize_text(clause)
    if not normalized_clause:
        return None, 0.0

    normalized_terms = [(normalize_text(t), t) for t in terms if normalize_text(t)]
    if not normalized_terms:
        return None, 0.0

    matched_norm, score, _ = process.extractOne(
        normalized_clause,
        [t[0] for t in normalized_terms],
        scorer=scorer,
    )
    for norm, original in normalized_terms:
        if norm == matched_norm:
            return original, float(score)
    return None, float(score)


def _best_keyword_label(
    text: str,
    accepted: list[str],
    rejected: list[str],
) -> tuple[ClauseLabel, str | None]:
    """Longest keyword/phrase match wins; ties favour rejected terms."""
    candidates: list[tuple[int, ClauseLabel, str]] = []

    for kw in accepted:
        if _phrase_in_text(kw, text):
            candidates.append((len(kw), ClauseLabel.ACCEPTED, kw))

    for kw in rejected:
        if _phrase_in_text(kw, text):
            candidates.append((len(kw), ClauseLabel.REJECTED, kw))

    if not candidates:
        return ClauseLabel.UNKNOWN, None

    max_len = max(c[0] for c in candidates)
    top = [c for c in candidates if c[0] == max_len]
    if any(c[1] == ClauseLabel.REJECTED for c in top):
        _, _, term = next(c for c in top if c[1] == ClauseLabel.REJECTED)
        return ClauseLabel.REJECTED, term
    _, _, term = next(c for c in top if c[1] == ClauseLabel.ACCEPTED)
    return ClauseLabel.ACCEPTED, term


def classify_clause(
    clause: str,
    accepted: list[str],
    rejected: list[str],
    *,
    accept_threshold: float = DEFAULT_ACCEPT_THRESHOLD,
    reject_threshold: float = DEFAULT_REJECT_THRESHOLD,
    scorer=DEFAULT_SCORER,
) -> ClauseMatch:
    normalized = normalize_text(clause)
    if not normalized:
        return ClauseMatch(clause=clause, label=ClauseLabel.UNKNOWN, score=0.0, matched_term=None)

    kw_label, kw_term = _best_keyword_label(normalized, accepted, rejected)
    if kw_label == ClauseLabel.REJECTED:
        return ClauseMatch(clause, ClauseLabel.REJECTED, 100.0, kw_term)
    if kw_label == ClauseLabel.ACCEPTED:
        return ClauseMatch(clause, ClauseLabel.ACCEPTED, 100.0, kw_term)

    accepted_norm = {normalize_text(t): t for t in accepted}
    rejected_norm = {normalize_text(t): t for t in rejected}

    if normalized in rejected_norm:
        return ClauseMatch(clause, ClauseLabel.REJECTED, 100.0, rejected_norm[normalized])
    if normalized in accepted_norm:
        return ClauseMatch(clause, ClauseLabel.ACCEPTED, 100.0, accepted_norm[normalized])

    accept_term, accept_score = _best_term_match(clause, accepted, scorer=scorer)
    reject_term, reject_score = (
        _best_term_match(clause, rejected, scorer=scorer) if rejected else (None, 0.0)
    )

    accept_hit = accept_score >= accept_threshold
    reject_hit = reject_score >= reject_threshold if rejected else False

    if accept_hit and reject_hit:
        if reject_score > accept_score:
            return ClauseMatch(clause, ClauseLabel.REJECTED, reject_score, reject_term)
        if accept_score > reject_score:
            return ClauseMatch(clause, ClauseLabel.ACCEPTED, accept_score, accept_term)
        return ClauseMatch(clause, ClauseLabel.REJECTED, reject_score, reject_term)
    if reject_hit:
        return ClauseMatch(clause, ClauseLabel.REJECTED, reject_score, reject_term)
    if accept_hit:
        return ClauseMatch(clause, ClauseLabel.ACCEPTED, accept_score, accept_term)

    return ClauseMatch(clause, ClauseLabel.UNKNOWN, max(accept_score, reject_score), None)


def _response_has_accepted(
    text: str,
    accepted: list[str],
    rejected: list[str],
    **kwargs,
) -> ClauseMatch | None:
    kw_label, kw_term = _best_keyword_label(text, accepted, rejected)
    if kw_label == ClauseLabel.ACCEPTED:
        return ClauseMatch(text, ClauseLabel.ACCEPTED, 100.0, kw_term)
    if kw_label == ClauseLabel.REJECTED:
        return None

    match = classify_clause(text, accepted, rejected, **kwargs)
    if match.label == ClauseLabel.ACCEPTED:
        return match
    return None


def split_join_clauses(text: str) -> list[str]:
    parts = COMBINER_RE.split(text)
    return [p.strip(" ,;") for p in parts if p.strip(" ,;")]


def split_or_clauses(text: str) -> list[str]:
    parts = re.split(r"\s+or\s+|,|;|/|\n", text, flags=re.IGNORECASE)
    return [p.strip(" ,;") for p in parts if p.strip(" ,;")]


def _has_combiner(text: str) -> bool:
    return COMBINER_RE.search(text) is not None


def _score_join_path(
    normalized: str,
    accepted: list[str],
    rejected: list[str],
    **kwargs,
) -> ConditionScoreResult:
    clauses = split_join_clauses(normalized)
    matches = [classify_clause(c, accepted, rejected, **kwargs) for c in clauses]
    accepted_matches = [m for m in matches if m.label == ClauseLabel.ACCEPTED]
    has_accepted = bool(accepted_matches)
    has_not_accepted = any(m.label != ClauseLabel.ACCEPTED for m in matches)

    if has_accepted and has_not_accepted:
        best = max(accepted_matches, key=lambda m: m.score)
        return ConditionScoreResult(
            fuzzy_correct=0,
            best_score=best.score,
            matched_term=best.matched_term,
            rule="join_accepted_with_not_accepted",
            clause_matches=matches,
        )

    if has_accepted:
        best = max(accepted_matches, key=lambda m: m.score)
        return ConditionScoreResult(
            fuzzy_correct=1,
            best_score=best.score,
            matched_term=best.matched_term,
            rule="join_accepted_only",
            clause_matches=matches,
        )

    if any(m.label == ClauseLabel.REJECTED for m in matches):
        return ConditionScoreResult(
            fuzzy_correct=0,
            best_score=max(m.score for m in matches),
            matched_term=None,
            rule="join_rejected_only",
            clause_matches=matches,
        )

    return ConditionScoreResult(
        fuzzy_correct=0,
        best_score=max((m.score for m in matches), default=0.0),
        matched_term=None,
        rule="join_no_accepted_match",
        clause_matches=matches,
    )


def score_condition_response(
    response: object,
    vignette: str,
    *,
    accept_threshold: float = DEFAULT_ACCEPT_THRESHOLD,
    reject_threshold: float = DEFAULT_REJECT_THRESHOLD,
    scorer=DEFAULT_SCORER,
) -> ConditionScoreResult:
    normalized = normalize_text(response)
    if not normalized:
        return ConditionScoreResult(
            fuzzy_correct=None,
            best_score=float("nan"),
            matched_term=None,
            rule="empty",
            clause_matches=[],
        )

    if vignette not in CASE_CONFIG:
        raise KeyError(f"Unknown vignette {vignette!r}")

    accepted = CASE_CONFIG[vignette]["accepted"]
    rejected = CASE_CONFIG[vignette].get("rejected", [])
    kwargs = {
        "accept_threshold": accept_threshold,
        "reject_threshold": reject_threshold,
        "scorer": scorer,
    }

    if _has_combiner(normalized):
        return _score_join_path(normalized, accepted, rejected, **kwargs)

    clauses = split_or_clauses(normalized) or [normalized]
    matches = [classify_clause(c, accepted, rejected, **kwargs) for c in clauses]
    accepted_matches = [m for m in matches if m.label == ClauseLabel.ACCEPTED]
    if accepted_matches:
        best = max(accepted_matches, key=lambda m: m.score)
        return ConditionScoreResult(
            fuzzy_correct=1,
            best_score=best.score,
            matched_term=best.matched_term,
            rule="any_accepted_or_listed",
            clause_matches=matches,
        )

    full_match = _response_has_accepted(normalized, accepted, rejected, **kwargs)
    if full_match:
        return ConditionScoreResult(
            fuzzy_correct=1,
            best_score=full_match.score,
            matched_term=full_match.matched_term,
            rule="keyword_in_sentence",
            clause_matches=[full_match],
        )

    return ConditionScoreResult(
        fuzzy_correct=0,
        best_score=max((m.score for m in matches), default=0.0),
        matched_term=None,
        rule="no_accepted_match",
        clause_matches=matches,
    )


## Set matching thresholds

In [ ]:
ACCEPT_THRESHOLD = 82
REJECT_THRESHOLD = 82
SCORER = fuzz.token_sort_ratio

print("Configured vignettes:")
for vignette in VIGNETTES:
    cfg = CASE_CONFIG[vignette]
    n_rejected = len(cfg.get("rejected", []))
    print(
        f"  {vignette}: {len(cfg['accepted'])} accepted, "
        f"{n_rejected} rejected terms"
    )

## Load data

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

needed_cols = [ID_COL, *TEXT_COLS.values(), *MANUAL_COLS.values()]
df = pd.read_excel(DATA_PATH)

missing_cols = [col for col in needed_cols if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}")

print(f"Loaded {len(df):,} participants from {DATA_PATH.name}")

## Apply automated scoring

In [ ]:
def apply_fuzzy_scoring(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    for vignette in VIGNETTES:
        for phase in PHASES:
            text_col = TEXT_COLS[(vignette, phase)]
            fuzzy_col = FUZZY_COLS[(vignette, phase)]
            score_col = f"vig_{vignette}_{phase}_condition_fuzzy_score"
            ref_col = f"vig_{vignette}_{phase}_condition_fuzzy_best_ref"
            rule_col = f"vig_{vignette}_{phase}_condition_fuzzy_rule"

            results: list[ConditionScoreResult] = [
                score_condition_response(
                    text,
                    vignette,
                    accept_threshold=ACCEPT_THRESHOLD,
                    reject_threshold=REJECT_THRESHOLD,
                    scorer=SCORER,
                )
                for text in out[text_col]
            ]

            out[fuzzy_col] = [r.fuzzy_correct for r in results]
            out[score_col] = [r.best_score for r in results]
            out[ref_col] = [r.matched_term for r in results]
            out[rule_col] = [r.rule for r in results]

    return out


df_fuzzy = apply_fuzzy_scoring(df)
print(f"Scored {len(df_fuzzy):,} participants ({df_fuzzy.filter(like='fuzzy_correct').shape[1]} fuzzy columns)")

## Compare fuzzy vs manual scoring

In [ ]:
def compare_manual_vs_fuzzy(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    disagreement_rows = []

    for vignette in VIGNETTES:
        for phase in PHASES:
            manual_col = MANUAL_COLS[(vignette, phase)]
            fuzzy_col = FUZZY_COLS[(vignette, phase)]
            text_col = TEXT_COLS[(vignette, phase)]
            rule_col = f"vig_{vignette}_{phase}_condition_fuzzy_rule"
            score_col = f"vig_{vignette}_{phase}_condition_fuzzy_score"

            sub = data[[ID_COL, text_col, manual_col, fuzzy_col, score_col, rule_col]].copy()
            comparable = sub[sub[manual_col].notna() & sub[fuzzy_col].notna()]

            n_compare = len(comparable)
            n_agree = int((comparable[manual_col] == comparable[fuzzy_col]).sum()) if n_compare else 0

            summary_rows.append(
                {
                    "vignette": vignette,
                    "phase": phase,
                    "n_compared": n_compare,
                    "n_agree": n_agree,
                    "pct_agree": round(n_agree / n_compare * 100, 2) if n_compare else float("nan"),
                    "n_manual_missing": int(sub[manual_col].isna().sum()),
                    "n_fuzzy_missing": int(sub[fuzzy_col].isna().sum()),
                }
            )

            disagree = comparable[comparable[manual_col] != comparable[fuzzy_col]]
            for row in disagree.itertuples(index=False):
                disagreement_rows.append(
                    {
                        ID_COL: getattr(row, ID_COL),
                        "vignette": vignette,
                        "phase": phase,
                        "text": getattr(row, text_col),
                        "manual_correct": getattr(row, manual_col),
                        "fuzzy_correct": getattr(row, fuzzy_col),
                        "fuzzy_score": getattr(row, score_col),
                        "fuzzy_rule": getattr(row, rule_col),
                    }
                )

    return pd.DataFrame(summary_rows), pd.DataFrame(disagreement_rows)


agreement_summary, disagreements = compare_manual_vs_fuzzy(df_fuzzy)
print(agreement_summary.to_string(index=False))
print(f"\nOverall agreement: {agreement_summary['n_agree'].sum():,} / {agreement_summary['n_compared'].sum():,}")
print(f"Disagreements to review: {len(disagreements):,}")

## Export manual–automated disagreements

In [ ]:
DISAGREEMENTS_PATH = ROOT / "data" / "condition_manual_vs_fuzzy_disagreements.xlsx"

export_df = disagreements.sort_values(["vignette", "phase", ID_COL]).reset_index(drop=True)
export_df.to_excel(DISAGREEMENTS_PATH, index=False)

print(f"Saved {len(export_df):,} disagreements to {DISAGREEMENTS_PATH}")

## Export agreement summary

In [ ]:
from docx import Document

AGREEMENT_DOC_PATH = ROOT / "data" / "condition_fuzzy_agreement_summary.docx"

COLUMN_LABELS = {
    "vignette": "Vignette",
    "phase": "Phase",
    "n_compared": "n compared",
    "n_agree": "n agree",
    "pct_agree": "% agree",
    "n_manual_missing": "n manual missing",
    "n_fuzzy_missing": "n fuzzy missing",
}

doc = Document()
doc.add_heading("Fuzzy vs manual condition scoring — agreement summary", level=1)

table_df = agreement_summary.copy()
headers = [COLUMN_LABELS.get(col, col) for col in table_df.columns]
table = doc.add_table(rows=1 + len(table_df), cols=len(headers))
table.style = "Table Grid"

for col_idx, label in enumerate(headers):
    table.rows[0].cells[col_idx].text = label

for row_idx, row in enumerate(table_df.itertuples(index=False), start=1):
    for col_idx, value in enumerate(row):
        col_name = table_df.columns[col_idx]
        if col_name == "pct_agree" and pd.notna(value):
            text = f"{value:.2f}"
        elif pd.isna(value):
            text = ""
        else:
            text = str(value)
        table.rows[row_idx].cells[col_idx].text = text

total_compared = int(agreement_summary["n_compared"].sum())
total_agree = int(agreement_summary["n_agree"].sum())
pct_overall = total_agree / total_compared * 100 if total_compared else float("nan")

doc.add_paragraph("")
doc.add_paragraph(
    f"Overall agreement: {total_agree:,} / {total_compared:,} ({pct_overall:.2f}%). "
    f"Disagreements: {len(disagreements):,}."
)

AGREEMENT_DOC_PATH.parent.mkdir(parents=True, exist_ok=True)
doc.save(AGREEMENT_DOC_PATH)

print(f"Saved agreement summary table to {AGREEMENT_DOC_PATH}")